# GenSpark — YOLOv8 Component Detector (Training on Colab)

Trains a PC-component object detector on Colab's **free GPU**, then lets you
download `best.pt` to deploy in the GenSpark backend.

## How to use
1. **Runtime → Change runtime type → T4 GPU → Save**  (free GPU)
2. In the **Dataset** cell, paste your Roboflow snippet (Option A) *or* upload your own (Option B).
3. **Runtime → Run all**.
4. The last cell downloads `best.pt` — send that file to your developer.

> Deploy target in the project: `backend/models/best.pt`. The detection code reads
> the model's own class names, so new classes (cpu/gpu/etc.) work automatically.


### 1. Check the GPU is on


In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE  ->  Runtime > Change runtime type > T4 GPU, then re-run')


### 2. Install YOLOv8 + Roboflow


In [ ]:
!pip install -q ultralytics roboflow


### 3. Get the dataset

**Option A — Roboflow (recommended, no manual labeling).**
On your Roboflow dataset page: *Download Dataset → Format: YOLOv8 → Show download code*,
then paste that snippet below (it already contains your API key).

Pick a dataset that has the classes you want (cpu, gpu, ram, motherboard, monitor, ...).


In [ ]:
# ---- OPTION A: paste your Roboflow snippet here (replace the 4 placeholder lines) ----
from roboflow import Roboflow
rf = Roboflow(api_key="PASTE_YOUR_API_KEY")
project = rf.workspace("WORKSPACE_ID").project("PROJECT_ID")
dataset = project.version(1).download("yolov8")

DATA_YAML = dataset.location + "/data.yaml"
print("Dataset ready. data.yaml =", DATA_YAML)


**Option B — upload your own dataset** (YOLOv8 format zip: `images/` + `labels/` + `data.yaml`).
Skip this if you used Option A.


In [ ]:
# ---- OPTION B (only if NOT using Roboflow) — uncomment to use ----
# from google.colab import files
# up = files.upload()                      # choose your dataset .zip
# import os, zipfile
# z = next(iter(up))
# zipfile.ZipFile(z).extractall('/content/mydata')
# DATA_YAML = '/content/mydata/data.yaml'  # adjust if data.yaml is in a subfolder
# print('Using', DATA_YAML)


### 4. Train  (≈15–30 min on the free GPU)
Starts from the pretrained `yolov8s.pt` so it learns fast and generalizes better.


In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')          # pretrained base
results = model.train(
    data=DATA_YAML,
    epochs=80,
    imgsz=640,
    batch=16,
    patience=20,                    # early-stop if no improvement
    name='genspark',
)
BEST = f'{results.save_dir}/weights/best.pt'
print('Trained weights:', BEST)


### 5. Check accuracy + the classes it learned


In [ ]:
m = YOLO(BEST)
print('Classes the model can now detect:', m.names)
metrics = m.val(data=DATA_YAML)
print('mAP@50 :', round(float(metrics.box.map50), 3))
print('mAP@50-95 :', round(float(metrics.box.map), 3))


### 6. (Optional) Quick visual test on a few validation images


In [ ]:
import glob, os
base = os.path.dirname(DATA_YAML)
imgs = (glob.glob(base + '/valid/images/*') or glob.glob(base + '/val/images/*')
        or glob.glob(base + '/test/images/*'))[:4]
if imgs:
    YOLO(BEST).predict(imgs, save=True, conf=0.35)
    print('Annotated previews saved under runs/  (open them from the file browser)')
else:
    print('No val/test images found to preview — that is fine.')


### 7. Download `best.pt`
Give this file to your developer — it goes to `backend/models/best.pt`.


In [ ]:
from google.colab import files
print('Downloading best.pt ...')
files.download(BEST)
